In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
df = pd.read_csv("data/Saber11_2022.csv")
df.head()

,periodo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_genero,cole_jornada,cole_naturaleza,estu_genero,fami_cuartoshogar,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20224,URBANO,N,A,ACADÉMICO,MIXTO,UNICA,OFICIAL,F,Tres,...,Si,Si,Si,A1,56,51,52,46,59,262
1,20224,URBANO,N,A,ACADÉMICO,MIXTO,UNICA,OFICIAL,M,Dos,...,No,No,Si,A1,48,51,49,46,45,239
2,20224,URBANO,N,A,ACADÉMICO,MIXTO,TARDE,OFICIAL,F,Tres,...,No,No,Si,A-,42,45,34,39,55,216
3,20224,URBANO,N,A,ACADÉMICO,MIXTO,UNICA,OFICIAL,F,Dos,...,No,Si,No,A-,43,30,37,41,39,186
4,20224,URBANO,N,A,ACADÉMICO,MIXTO,MAÑANA,OFICIAL,F,Dos,...,No,No,Si,A-,43,62,50,54,66,284


Se eliminan las variables de los puntajes por asignaturas por tener alta correlacción

In [4]:
df = df.drop(['punt_matematicas', 'punt_sociales_ciudadanas', 'punt_c_naturales', 'punt_lectura_critica', 'punt_ingles'], axis = 1)
df.columns.tolist()

['periodo',
 'cole_area_ubicacion',
 'cole_bilingue',
 'cole_calendario',
 'cole_caracter',
 'cole_genero',
 'cole_jornada',
 'cole_naturaleza',
 'estu_genero',
 'fami_cuartoshogar',
 'fami_educacionmadre',
 'fami_educacionpadre',
 'fami_estratovivienda',
 'fami_personashogar',
 'fami_tieneautomovil',
 'fami_tienecomputador',
 'fami_tieneinternet',
 'fami_tienelavadora',
 'desemp_ingles',
 'punt_global']

In [ ]:
X = df.drop('punt_global', axis = 1)
y = df.punt_global
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=123)

In [3]:
#Importe MLFlow para registrar los experimentos, el regresor de bosques aleatorios y la métrica de error cuadrático medio
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

c:\Users\kevin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def configure_experiment(name_experiment, modelos, option):
    # defina el servidor para llevar el registro de modelos y artefactos
    mlflow.set_tracking_uri("http://0.0.0.0:5000")

    # registre el experimento
    experiment = mlflow.set_experiment(name_experiment)

    if option == 1:
        rf = modelos["Regresion_Lineal"]
        nombre_modelo = 'regresion-lineal-model'

    elif option == 2:
        rf = modelos["Random_Forest"]
        nombre_modelo = 'random-forest-model'
    
    elif option == 3:
        rf = modelos["Gradient_Boosting"]
        nombre_modelo = 'gradient-boosting-model'

    else:
        raise ValueError("Opcion No Válida")

    # Aquí se ejecuta MLflow sin especificar un nombre o id del experimento. MLflow los crea un experimento para este cuaderno por defecto y guarda las características del experimento y las métricas definidas. 
    # Para ver el resultado de las corridas haga click en Experimentos en el menú izquierdo. 
    with mlflow.start_run(experiment_id=experiment.experiment_id):
        # mlflow.set_tag("modelo", nombre_modelo)

        mlflow.log_params(rf.get_params())
        
        # Cree el modelo con los parámetros definidos y entrénelo
        rf.fit(X_train, y_train)
        # Realice predicciones de prueba
        predictions = rf.predict(X_test)
    
        # Registre el modelo
        mlflow.sklearn.log_model(rf, 'modelo')
    
        # Cree y registre la métrica de interés
        mse = mean_squared_error(y_test, predictions)
        mlflow.log_metric("mse", mse)
        print(f"modelo: {nombre_modelo}")
        print(f"MSE: {mse}")

In [ ]:
modelos = {
    "Regresion_Lineal": LinearRegression(),

    "Random_Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=6,
        max_features=4,
        random_state=42
    ),

    "Gradient_Boosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    )
}

In [ ]:
configure_experiment(
    "Regresion_Lineal",
    modelos,
    option=1
)

In [ ]:
configure_experiment(
    "Random_Forest",
    modelos,
    option=2
)

In [ ]:
configure_experiment(
    "Gradient_Boosting",
    modelos,
    option=3
)